# Separable Weights Demo

This notebook demonstrates the separable weights feature for multi-head MobileNet V3 models.

## Features Demonstrated:
1. **Setup & Model Creation** - Create model with `separable_weights=True`
2. **Separate Weight Saving/Loading** - Save/load backbone and heads independently
3. **Feature Caching** - Extract features once, reuse with multiple heads
4. **Dynamic Head Operations** - Load/unload heads, add new heads
5. **Training Control** - Freeze backbone/heads for transfer learning
6. **Memory & Performance** - Compare full model vs cached features
7. **Real Inference** - End-to-end example with images


## 1. Setup & Model Creation


In [ ]:
import sys
from pathlib import Path
import numpy as np
import tensorflow as tf

# Add project root to path
project_root = Path().absolute().parent
sys.path.insert(0, str(project_root))

# Import model components
from models.components.multi_head_model_config import MultiHeadModelConfig
from models.components.head_configuration import create_head_config_from_list
from models.architectures.mobilenet_v3_qat_multi import MultiHeadMobileNetV3QATArchitecture
from models.utils import FeatureCacheManager

print(f"TensorFlow version: {tf.__version__}")
print(f"Project root: {project_root}")


In [ ]:
# Create model with separable_weights enabled
head_configs = create_head_config_from_list(
    [5, 2, 3],  # 5 classes, 2 classes, 3 classes
    head_names=["object_class", "person_detection", "age_group"]
)

config = MultiHeadModelConfig(
    input_shape=(128, 128, 3),
    head_configs=head_configs,
    arch_params={
        'alpha': 0.25,
        'use_pretrained': False,
    },
    training_mode='joint',
    separable_weights=True  # Enable separable weights
)

# Create architecture
architecture = MultiHeadMobileNetV3QATArchitecture(config)
model = architecture.get_model()

print(f"Model: {architecture.name}")
print(f"Separable weights enabled: {architecture.is_separable}")
print(f"Heads: {architecture.head_names}")
print(f"Total parameters: {model.count_params():,}")


## 2. Separate Weight Saving/Loading


In [ ]:
import tempfile
import os

# Create temporary directory for weights
weights_dir = tempfile.mkdtemp(prefix="separable_weights_demo_")
print(f"Weights directory: {weights_dir}")

# Save all weights separately using convenience method
saved_files = architecture.save_all_weights_separately(weights_dir)

# List saved files
print("\nSaved weight files:")
for name, path in saved_files.items():
    size_kb = os.path.getsize(path) / 1024
    print(f"  {name}: {size_kb:.1f} KB")


In [ ]:
# Create a new architecture and load weights separately
config2 = MultiHeadModelConfig(
    input_shape=(128, 128, 3),
    head_configs=create_head_config_from_list([5, 2, 3], ["object_class", "person_detection", "age_group"]),
    arch_params={'alpha': 0.25, 'use_pretrained': False},
    separable_weights=True
)

architecture2 = MultiHeadMobileNetV3QATArchitecture(config2)
model2 = architecture2.get_model()

# Load all weights at once
architecture2.load_all_weights_separately(weights_dir)

# Verify weights are the same by comparing predictions
test_input = np.random.rand(1, 128, 128, 3).astype(np.float32)
pred1 = model.predict(test_input, verbose=0)
pred2 = model2.predict(test_input, verbose=0)

print("\nPrediction comparison (should be identical):")
for head_name in architecture.head_names:
    diff = np.max(np.abs(pred1[head_name] - pred2[head_name]))
    print(f"  {head_name}: Max difference = {diff:.6f}")


## 3. Feature Caching


In [ ]:
# Create feature cache manager
cache_manager = FeatureCacheManager(architecture)

print(f"Cache Manager: {cache_manager}")
print(f"Feature shape: {cache_manager.feature_shape}")

# Generate sample images
batch_size = 16
sample_images = np.random.rand(batch_size, 128, 128, 3).astype(np.float32)

# Extract and cache features
features = cache_manager.extract_features(sample_images)

print(f"\nInput shape: {sample_images.shape}")
print(f"Feature shape: {features.shape}")
print(f"Features cached: {cache_manager.has_cached_features}")


In [ ]:
# Save features to disk
features_path = os.path.join(weights_dir, "cached_features.npy")
cache_manager.save_features(features_path)

# Clear cache
cache_manager.clear_cache()
print(f"Features cached: {cache_manager.has_cached_features}")

# Load features from disk
loaded_features = cache_manager.load_features(features_path)
print(f"Features loaded: {loaded_features.shape}")
print(f"Features cached: {cache_manager.has_cached_features}")


## 4. Dynamic Head Operations


In [ ]:
# Load a head dynamically
cache_manager.load_head("object_class")

# Run prediction using cached features
predictions = cache_manager.predict_with_cached_features("object_class")
print(f"Predictions shape: {predictions.shape}")
print(f"Sample predictions (first 3 samples):")
print(predictions[:3])


In [ ]:
# Load multiple heads
cache_manager.load_head("person_detection")
cache_manager.load_head("age_group")

print(f"Loaded heads: {cache_manager.loaded_head_names}")

# Run all heads at once
all_predictions = cache_manager.predict_all_loaded_heads()

print("\nPredictions from all heads:")
for head_name, preds in all_predictions.items():
    print(f"  {head_name}: {preds.shape} - Max prob: {preds.max():.3f}")


In [ ]:
# Unload heads to free memory
cache_manager.unload_head("age_group")
print(f"Loaded heads after unload: {cache_manager.loaded_head_names}")

cache_manager.unload_all_heads()
print(f"Loaded heads after unload_all: {cache_manager.loaded_head_names}")


In [ ]:
# Add a new head dynamically
print(f"Current heads: {architecture.head_names}")

new_model = architecture.add_head_dynamically(
    num_classes=4,
    head_name="emotion",
    activation='linear',
    freeze_backbone=True
)

print(f"\nHeads after adding: {architecture.head_names}")
print(f"New model output names: {new_model.output_names}")


## 5. Training Control with Frozen Components


In [ ]:
# Check trainable status
status = architecture.get_trainable_status()
print("Trainable status:")
for component, trainable in status.items():
    print(f"  {component}: {'Trainable' if trainable else 'Frozen'}")


In [ ]:
# Freeze backbone for transfer learning
architecture.freeze_backbone()

# Freeze existing heads (keep only emotion trainable)
architecture.freeze_head("object_class")
architecture.freeze_head("person_detection")
architecture.freeze_head("age_group")

# Check status again
status = architecture.get_trainable_status()
print("Trainable status after freezing:")
for component, trainable in status.items():
    print(f"  {component}: {'Trainable' if trainable else 'Frozen'}")


In [ ]:
# Create synthetic training data for the new head
num_samples = 100
train_images = np.random.rand(num_samples, 128, 128, 3).astype(np.float32)
train_labels = {
    'object_class': np.random.randint(0, 5, num_samples),
    'person_detection': np.random.randint(0, 2, num_samples),
    'age_group': np.random.randint(0, 3, num_samples),
    'emotion': np.random.randint(0, 4, num_samples)  # New head
}

# Compile model for training
# Note: Only the new head (emotion) will be trained since others are frozen
new_model.compile(
    optimizer='adam',
    loss={
        head: tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
        for head in new_model.output_names if head != 'unified_heads'
    },
    loss_weights={
        'object_class': 0.0,      # Frozen, don't contribute to loss
        'person_detection': 0.0,  # Frozen
        'age_group': 0.0,         # Frozen
        'emotion': 1.0            # Train only this head
    }
)

print("Model compiled. Only 'emotion' head will be trained.")


In [ ]:
# Train for a few epochs (only the emotion head will be updated)
history = new_model.fit(
    train_images,
    train_labels,
    epochs=3,
    batch_size=16,
    verbose=1
)

print("\nTraining complete!")
print("Note: Backbone and other heads remained frozen.")


## 6. Memory & Performance Comparison


In [ ]:
# Get memory usage information
cache_manager = FeatureCacheManager(architecture)

# Extract features
test_batch = np.random.rand(32, 128, 128, 3).astype(np.float32)
features = cache_manager.extract_features(test_batch)

# Load all heads
for head_name in architecture.head_names:
    cache_manager.load_head(head_name)

# Get memory usage
memory_info = cache_manager.get_memory_usage()

print("Memory Usage:")
print(f"  Cached features: {memory_info['cached_features']['size_mb']:.2f} MB")
print(f"  Backbone: {memory_info['backbone']['parameters']:,} params ({memory_info['backbone']['size_mb']:.2f} MB)")
print(f"  Loaded heads: {memory_info['total_heads_loaded']}")
for head_name, info in memory_info['loaded_heads'].items():
    print(f"    {head_name}: {info['parameters']:,} params ({info['estimated_size_mb']:.3f} MB)")
print(f"  Total memory: {memory_info['summary']['total_memory_mb']:.2f} MB")


## 7. Real Inference Examples


In [ ]:
# Create a realistic inference example
# Simulate 5 images
num_images = 5
images = np.random.rand(num_images, 128, 128, 3).astype(np.float32)

# Create fresh cache manager
inference_cache = FeatureCacheManager(architecture)

print("Step 1: Extract features from images (one-time)")
features = inference_cache.extract_features(images)
print(f"  Features extracted: {features.shape}")


In [ ]:
print("Step 2: Load and run each head")

results = {}

for head_name in ['object_class', 'person_detection', 'age_group', 'emotion']:
    # Load head
    inference_cache.load_head(head_name)
    
    # Run prediction
    predictions = inference_cache.predict_with_cached_features(head_name)
    
    # Store results
    results[head_name] = predictions
    
    # Get class predictions
    class_predictions = np.argmax(predictions, axis=1)
    confidences = np.max(predictions, axis=1)
    
    print(f"\n  {head_name}:")
    for i in range(num_images):
        print(f"    Image {i+1}: Class {class_predictions[i]}, Confidence: {confidences[i]:.3f}")
    
    # Unload head to free memory
    inference_cache.unload_head(head_name)


In [ ]:
print("Step 3: Summary")
print("="*50)
print(f"Images processed: {num_images}")
print(f"Heads used: {list(results.keys())}")
print(f"Features extracted once: {features.shape}")
print(f"Predictions per head: {predictions.shape}")
print("="*50)
print("\nKey benefit: Features were extracted only once,")
print("then reused for all 4 classification heads.")
print("This saves computation when running multiple heads.")


## Cleanup


In [ ]:
# Clean up temporary files
import shutil
shutil.rmtree(weights_dir)
print(f"Cleaned up: {weights_dir}")


## Summary

This notebook demonstrated:

1. **Separable Weights**: Enable with `separable_weights=True` in config
2. **Save/Load Backbone & Heads**: Use `save_backbone_weights()`, `load_backbone_weights()`, etc.
3. **Feature Caching**: Use `FeatureCacheManager` to extract features once and reuse
4. **Dynamic Heads**: Load/unload heads with `load_head()`, `unload_head()`
5. **Add New Heads**: Use `add_head_dynamically()` to extend models
6. **Training Control**: Freeze backbone/heads with `.trainable = False` or helper methods

These features enable:
- Transfer learning with frozen backbones
- Extending models with new classification tasks
- Memory-efficient inference with feature caching
- Modular weight management for deployment
